# quant-validator — Quick Start

This notebook shows how to validate a trading strategy in 30 seconds.

**What you get:** a verdict on whether your strategy's edge is real or noise.

## 1. Install the toolkit

In [ ]:
!pip install -q git+https://github.com/simonemasciaquant/quant-validator.git

## 2. Generate example trades

For this demo, we generate 2000 synthetic trades with a small positive edge. In real usage, you would load your own trades from CSV with columns: `asset`, `entry_ts`, `exit_ts`, `ret_net`.

In [ ]:
import numpy as np
import pandas as pd
from quant_validator.core import Validator

# Generate synthetic trades with a small positive edge
np.random.seed(42)
n = 2000
dates = pd.date_range('2023-01-01', periods=n, freq='6h')
assets = ['BTC', 'ETH', 'SOL', 'AVAX', 'LINK']

trades = pd.DataFrame({
    'asset': np.random.choice(assets, n),
    'entry_ts': dates,
    'exit_ts': dates + pd.Timedelta(hours=4),
    'ret_net': np.random.normal(0.0005, 0.02, n),
})

print(f'Generated {len(trades)} trades')
trades.head()

## 3. Run the validation

One function call runs all tests: basic metrics, placebo, LOAO, walk-forward, and Deflated Sharpe Ratio.

In [ ]:
v = Validator(
    trades,
    asset_col='asset',
    entry_col='entry_ts',
    exit_col='exit_ts',
    ret_col='ret_net',
    capital=10000,
    max_conc=10,
    periods_per_year=365.0,
)

v.run_all(split_date='2024-01-01', n_trials=1)

## 4. Read the verdict

In [ ]:
print(v.summary())

## 5. What the results mean

The report gives you:

- **Basic metrics**: Sharpe (calendar), Sharpe (active), Max DD, PF, Hit rate
- **Placebo test**: does the direction of the signal have predictive power?
- **LOAO**: is the edge concentrated in one asset?
- **Walk-forward**: does the strategy survive out-of-sample?
- **DSR**: is the Sharpe significant after correcting for multiple testing?

Each test gives a PASS or FAIL verdict. A strategy passes validation only if **all tests pass**.

## Use with your own data

Replace the synthetic trades with your own:

```python
trades = pd.read_csv('my_trades.csv')
v = Validator(trades)
v.run_all(split_date='2024-01-01', n_trials=8)
print(v.summary())
```

## Disclaimer

This toolkit does NOT confirm that a strategy will be profitable. It only tests whether the backtest is statistically sound. A strategy can pass all tests and still lose money.

---

**GitHub:** https://github.com/simonemasciaquant/quant-validator
